<a href="https://colab.research.google.com/github/eoade/Umbrella-Recommendation/blob/main/Umbrella_Recommendation_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# 1. Create a sample dataset with missing weather data (NaN represents missing rainfall info)
data = {
    'Day': ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'],
    'Cloud_Cover_%': [80, 90, 30, 75, 85, 20, 95],
    'Rainfall_mm': [5.2, np.nan, 0.0, np.nan, 12.0, 0.0, np.nan]  # Missing data on Tue, Thu, Sun
}

df = pd.DataFrame(data)
print("Original incomplete weather data:")
print(df)
print("\n" + "="*40 + "\n")

# 2. Fill missing gaps with estimates
# We will estimate missing rainfall based on a combination of forward fill (recent trend)
# and the overall average rainfall for cloudy days, or simple interpolation.
# For simplicity and robustness, we will interpolate or fill based on Cloud Cover proxy.

# Let's fill using linear interpolation as a first estimate
df['Estimated_Rainfall_mm'] = df['Rainfall_mm'].interpolate(method='linear').fillna(0.0)

# If we still have NaN (e.g., at the start), we fill with 0.0
df['Estimated_Rainfall_mm'] = df['Estimated_Rainfall_mm'].round(2)

# 3. Decision Logic for "Carry Umbrella" recommendation
# Rule: Carry an umbrella if the estimated/actual rainfall > 1.0mm OR Cloud Cover > 80%
def recommend_umbrella(row):
    if row['Estimated_Rainfall_mm'] > 1.0 or row['Cloud_Cover_%'] >= 80:
        return "Yes (Carry Umbrella)"
    else:
        return "No"

df['Umbrella_Recommendation'] = df.apply(recommend_umbrella, axis=1)

print("Processed data with estimates and recommendations:")
print(df[['Day', 'Cloud_Cover_%', 'Rainfall_mm', 'Estimated_Rainfall_mm', 'Umbrella_Recommendation']])

Original incomplete weather data:
         Day  Cloud_Cover_%  Rainfall_mm
0     Monday             80          5.2
1    Tuesday             90          NaN
2  Wednesday             30          0.0
3   Thursday             75          NaN
4     Friday             85         12.0
5   Saturday             20          0.0
6     Sunday             95          NaN


Processed data with estimates and recommendations:
         Day  Cloud_Cover_%  Rainfall_mm  Estimated_Rainfall_mm  \
0     Monday             80          5.2                    5.2   
1    Tuesday             90          NaN                    2.6   
2  Wednesday             30          0.0                    0.0   
3   Thursday             75          NaN                    6.0   
4     Friday             85         12.0                   12.0   
5   Saturday             20          0.0                    0.0   
6     Sunday             95          NaN                    0.0   

  Umbrella_Recommendation  
0    Yes (Carry Umbr

In [ ]:
import requests
import pandas as pd
from google.colab import userdata

def get_osogbo_weather():
    # Retrieve your API key securely from Colab secrets
    try:
        api_key = userdata.get('OPENWEATHER_API_KEY')
    except Exception:
        print("Error: Please add your OPENWEATHER_API_KEY to Colab's secrets manager (the 🔑 icon in the left menu).")
        return None

    # Coordinates and API details for Osogbo, Osun State, Nigeria
    city = "Osogbo,NG"
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"

    response = requests.get(url)
    if response.status_code != 200:
        print(f"API request failed with status code {response.status_code}. Detail: {response.text}")
        return None

    data = response.json()

    # Extract relevant metrics
    cloud_cover = data.get('clouds', {}).get('all', 0)  # Cloudiness %

    # OpenWeather API might omit the 'rain' object if there is no current rain
    rain_dict = data.get('rain', {})
    rainfall = rain_dict.get('1h', 0.0)  # Rainfall volume for the last 1 hour, if available

    weather_desc = data.get('weather', [{}])[0].get('description', 'clear sky')
    temp = data.get('main', {}).get('temp')

    # Compile into a structure resembling our dataset
    real_weather = {
        'Location': 'Osogbo, Nigeria',
        'Description': weather_desc,
        'Temperature_C': temp,
        'Cloud_Cover_%': cloud_cover,
        'Rainfall_mm': rainfall
    }

    return pd.DataFrame([real_weather])

# Fetch and display the weather data
weather_df = get_osogbo_weather()
if weather_df is not None:
    # Apply our recommendation logic
    # Rule: Carry if estimated rainfall > 1.0mm OR cloud cover >= 80%
    weather_df['Umbrella_Recommendation'] = weather_df.apply(
        lambda row: "Yes (Carry Umbrella)" if row['Rainfall_mm'] > 1.0 or row['Cloud_Cover_%'] >= 80 else "No",
        axis=1
    )
    display(weather_df)

,Location,Description,Temperature_C,Cloud_Cover_%,Rainfall_mm,Umbrella_Recommendation
0,"Osogbo, Nigeria",broken clouds,29.56,84,0.0,Yes (Carry Umbrella)
